# ProQuest News Dataset Cleaning & Standardisation Pipeline
## *From Tweets to Trades: Analysing Sentiment Signal and Information Network for Crypto Market Prediction*

Cleans and standardises the ProQuest news article CSVs (one per asset query) produced by the web scraper. Each file is processed independently and saved separately, preserving the per-asset structure needed for separate sentiment analysis.

**Workflow:**
1. Upload all per-asset ProQuest CSVs at once.
2. Run cleaning on each file.
3. Download all cleaned files as a ZIP.

**Cleaning steps applied:**
- Remove rows with missing or failed-extraction text (e.g. literal 'Full text').
- Parse and standardise publication dates.
- Normalise article text (whitespace, quotes, unicode artefacts).
- Clean titles, source names, and author fields.
- Enforce minimum and maximum article length thresholds.
- Verify asset relevance against the target asset's keywords.
- Deduplicate by article URL and by exact title+source combination.

## 0. Setup & Dependencies

In [ ]:
!pip install -q pandas

import pandas as pd
import re
import os
import shutil
from datetime import datetime
from pathlib import Path
from google.colab import files

os.makedirs('data/raw/news', exist_ok=True)
os.makedirs('data/processed/news', exist_ok=True)
os.makedirs('data/processed/news/stats', exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Configuration

Defines the study period, text length thresholds appropriate for news articles, and the asset-specific relevance keywords used to verify each article actually concerns the target asset.

In [ ]:
# --- Study period ---
START_DATE = pd.Timestamp('2020-01-01')
END_DATE = pd.Timestamp('2024-12-31')

# --- Length thresholds for news articles ---
# Articles should have substantial text; the scraper truncated to 2000 chars.
MIN_TEXT_LENGTH = 150
MAX_TEXT_LENGTH = 5000

# --- Extraction artefacts ---
# Strings that indicate the full-text extraction failed. These rows
# should be dropped because they contain a UI label rather than content.
BAD_TEXT_MARKERS = [
    'full text',
    'full-text',
    'abstract not available',
    'no abstract available',
    'details available',
    'record not available',
]

# --- Asset-specific relevance keywords ---
# Used to verify the article actually discusses the target asset.
ASSET_KEYWORDS = {
    'bitcoin':        ['bitcoin', 'btc', 'satoshi'],
    'ethereum':       ['ethereum', 'ether', 'eth', 'vitalik'],
    'dogecoin':       ['dogecoin', 'doge'],
    'shiba_inu':      ['shiba inu', 'shib', 'shibarmy', 'shiba'],
    'crypto_general': ['crypto', 'cryptocurrency', 'bitcoin', 'ethereum',
                       'blockchain', 'digital asset', 'defi', 'altcoin'],
}

print(f'Study period: {START_DATE.date()} to {END_DATE.date()}')
print(f'Asset categories: {list(ASSET_KEYWORDS.keys())}')

Study period: 2020-01-01 to 2024-12-31
Asset categories: ['bitcoin', 'ethereum', 'dogecoin', 'shiba_inu', 'crypto_general']


In [ ]:
# --- Helper functions ---

def clean_text(text):
    """Normalise article text for sentiment analysis."""
    if not isinstance(text, str):
        return ''
    # Replace curly quotes with straight quotes.
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    # Replace em/en dashes with regular hyphens.
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    # Remove ellipsis artefacts and non-breaking spaces.
    text = text.replace('\u2026', '...').replace('\xa0', ' ')
    # Remove HTML entities if any slipped through.
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    # Collapse multiple whitespace chars (including newlines and tabs).
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_extraction_failure(text):
    """Detect rows where full-text extraction failed at the scraper stage."""
    if not text:
        return True
    text_lower = text.strip().lower()
    # Exact match to any bad marker means extraction failed.
    if text_lower in BAD_TEXT_MARKERS:
        return True
    # Very short text that contains only a bad marker is also a failure.
    if len(text_lower) < 30 and any(m in text_lower for m in BAD_TEXT_MARKERS):
        return True
    return False

def parse_date(date_str):
    """Parse ProQuest-style date strings into a standardised YYYY-MM-DD format."""
    if not date_str or not isinstance(date_str, str):
        return None
    # Try pandas' flexible parser first.
    try:
        parsed = pd.to_datetime(date_str, errors='coerce', utc=True)
        if pd.notna(parsed):
            return parsed.tz_localize(None) if parsed.tz else parsed
    except Exception:
        pass
    # Try common ProQuest patterns.
    cleaned = re.sub(r'[;,].*$', '', date_str).strip()
    for fmt in ['%b %d, %Y', '%B %d, %Y', '%d %b %Y', '%d %B %Y',
                '%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y']:
        try:
            return pd.to_datetime(cleaned, format=fmt)
        except (ValueError, TypeError):
            continue
    return None

def is_relevant(text, keywords):
    """Check if article text mentions the target asset."""
    if not text or not keywords:
        return False
    text_lower = text.lower()
    return any(kw in text_lower for kw in keywords)

def detect_asset_from_filename(filename):
    """Infer which asset a ProQuest CSV corresponds to."""
    fname_lower = filename.lower()
    # Order matters: check compound names before short ones.
    for asset in ['shiba_inu', 'shibainu', 'shibarmy', 'shib',
                  'crypto_general', 'crypto general', 'general',
                  'bitcoin', 'ethereum', 'dogecoin']:
        if asset in fname_lower:
            if asset in ('shibainu', 'shibarmy', 'shib'):
                return 'shiba_inu'
            if asset in ('crypto general', 'general'):
                return 'crypto_general'
            if asset == 'crypto_general':
                return 'crypto_general'
            return asset
    return None

print('Helper functions loaded.')

Helper functions loaded.


## 2. Upload All ProQuest CSVs at Once

Select all your ProQuest article CSVs (one per asset query). Hold Shift or Cmd/Ctrl to multi-select. Files are saved to `data/raw/news/`.

In [ ]:
print('Upload all your ProQuest article CSVs (one per asset query):\n')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'data/raw/news/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved: {dest} ({len(content)/1e6:.1f} MB)')

print(f'\n{len(uploaded)} file(s) uploaded.')

Upload all your ProQuest article CSVs (one per asset query):



Saving all_proquest_articles (1).csv to all_proquest_articles (1).csv
Saving bitcoin_articles (1).csv to bitcoin_articles (1).csv
Saving crypto_general_articles.csv to crypto_general_articles.csv
Saving dogecoin_articles.csv to dogecoin_articles.csv
Saving ethereum_articles.csv to ethereum_articles.csv
Saving shiba_inu_articles.csv to shiba_inu_articles.csv
  Saved: data/raw/news/all_proquest_articles (1).csv (2.9 MB)
  Saved: data/raw/news/bitcoin_articles (1).csv (1.8 MB)
  Saved: data/raw/news/crypto_general_articles.csv (0.5 MB)
  Saved: data/raw/news/dogecoin_articles.csv (0.1 MB)
  Saved: data/raw/news/ethereum_articles.csv (0.3 MB)
  Saved: data/raw/news/shiba_inu_articles.csv (0.2 MB)

6 file(s) uploaded.


## 3. Cleaning & Standardisation Function

Standardises column names, cleans text fields, parses dates, applies relevance and length filters, and deduplicates. Output schema is consistent across all asset files for easy downstream integration.

In [ ]:
def clean_proquest_file(input_path, output_path, stats_path, asset_keywords):
    print('\n' + '=' * 60)
    print(f'CLEANING: {os.path.basename(input_path)}')
    print('=' * 60)

    try:
        df = pd.read_csv(input_path, on_bad_lines='skip',
                         engine='python', encoding='utf-8',
                         encoding_errors='ignore')
    except Exception as e:
        print(f'  ERROR loading file: {e}')
        return None

    initial = len(df)
    print(f'  Loaded: {initial:,} articles')
    print(f'  Columns: {list(df.columns)}')

    # Standardise column names to lowercase for detection.
    col_map = {col: col.lower().strip() for col in df.columns}
    df = df.rename(columns=col_map)

    # Expected columns from the scraper: title, pub_date, source, author,
    # full_text, url, scrape_timestamp, query_name, month, date_from, date_to.
    required = ['title', 'full_text']
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f'  ERROR: Missing required columns: {missing}')
        return None

    stats = {k: 0 for k in ['initial', 'dropped_extraction_failure',
                             'dropped_length', 'dropped_date',
                             'dropped_irrelevant', 'dropped_duplicate', 'kept']}
    stats['initial'] = initial

    # Clean all text fields.
    for col in ['title', 'full_text', 'source', 'author']:
        if col in df.columns:
            df[col] = df[col].fillna('').astype(str).apply(clean_text)

    # 1. Drop rows where full-text extraction failed.
    b = len(df)
    df = df[~df['full_text'].apply(is_extraction_failure)]
    stats['dropped_extraction_failure'] = b - len(df)

    # 2. Length filter.
    b = len(df)
    lens = df['full_text'].str.len()
    df = df[(lens >= MIN_TEXT_LENGTH) & (lens <= MAX_TEXT_LENGTH)]
    stats['dropped_length'] = b - len(df)

    # 3. Parse and filter by date.
    if 'pub_date' in df.columns:
        df['parsed_date'] = df['pub_date'].apply(parse_date)
        # Fallback: use month field from scraper metadata if pub_date failed.
        if 'month' in df.columns:
            fallback = df['parsed_date'].isna() & df['month'].notna()
            df.loc[fallback, 'parsed_date'] = pd.to_datetime(
            df.loc[fallback, 'month'].astype(str) + '-01', errors='coerce')
        # Force conversion to a proper datetime column (this fixes mixed types).
        df['parsed_date'] = pd.to_datetime(df['parsed_date'], errors='coerce')
        b = len(df)
        df = df[df['parsed_date'].notna()]
        df = df[(df['parsed_date'] >= START_DATE) &
            (df['parsed_date'] <= END_DATE)]
        stats['dropped_date'] = b - len(df)
        # Standardise to YYYY-MM-DD string for consistency.
        df['date'] = df['parsed_date'].dt.strftime('%Y-%m-%d')

    # 4. Relevance filter: title + first 500 chars of body must mention asset.
    b = len(df)
    df['_check_text'] = (df['title'] + ' ' +
                         df['full_text'].str[:500])
    df = df[df['_check_text'].apply(lambda t: is_relevant(t, asset_keywords))]
    df = df.drop(columns=['_check_text'])
    stats['dropped_irrelevant'] = b - len(df)

    # 5. Deduplicate.
    b = len(df)
    # First by URL if available.
    if 'url' in df.columns:
        df = df.drop_duplicates(subset=['url'], keep='first')
    # Then by title + first 200 chars of text (catches same article via
    # different URL paths).
    df['_dedup_key'] = df['title'] + '||' + df['full_text'].str[:200]
    df = df.drop_duplicates(subset=['_dedup_key'], keep='first')
    df = df.drop(columns=['_dedup_key'])
    stats['dropped_duplicate'] = b - len(df)

    # Build the standardised output schema.
    output_cols = ['date', 'title', 'full_text', 'source', 'author',
                   'url', 'query_name', 'month']
    available = [c for c in output_cols if c in df.columns]
    df_out = df[available].reset_index(drop=True)
    df_out = df_out.sort_values('date' if 'date' in df_out.columns else 'title')

    stats['kept'] = len(df_out)
    df_out.to_csv(output_path, index=False)

    print(f'\n  --- Summary ---')
    for key, val in stats.items():
        print(f'  {key:30s} {val:>8,}')
    if initial > 0:
        print(f'  {"retention":30s} {100 * stats["kept"] / initial:>7.1f}%')
    if len(df_out) > 0 and 'date' in df_out.columns:
        print(f'  Date range: {df_out["date"].min()} to {df_out["date"].max()}')
    print(f'  Saved: {output_path}')

    if stats_path:
        with open(stats_path, 'w') as f:
            f.write(f'ProQuest Cleaning Statistics\nFile: {input_path}\n')
            f.write(f'Run: {datetime.now().isoformat()}\n' + '=' * 50 + '\n\n')
            for key, val in stats.items():
                f.write(f'{key}: {val:,}\n')

    return stats

print('Cleaning function loaded.')

Cleaning function loaded.


## 4. Process All Uploaded Files

Iterates through every uploaded ProQuest CSV, detects the asset from the filename, and applies the appropriate relevance keywords.

In [ ]:
INPUT_DIR = 'data/raw/news'
OUTPUT_DIR = 'data/processed/news'
STATS_DIR = 'data/processed/news/stats'

# Find uploaded CSVs, excluding any 'all_' master files.
csv_files = sorted([f for f in os.listdir(INPUT_DIR)
                    if f.endswith('.csv') and not f.startswith('all_')])

if not csv_files:
    print(f'No CSV files found in {INPUT_DIR}. Upload files in Step 2 first.')
else:
    print('=' * 60)
    print(f'PROCESSING {len(csv_files)} PROQUEST FILE(S)')
    print('=' * 60)
    for f in csv_files:
        size_mb = os.path.getsize(os.path.join(INPUT_DIR, f)) / 1e6
        print(f'  {f} ({size_mb:.1f} MB)')

    overall_stats = {}
    for fname in csv_files:
        input_path = os.path.join(INPUT_DIR, fname)
        output_path = os.path.join(OUTPUT_DIR, fname.replace('.csv', '_cleaned.csv'))
        stats_path = os.path.join(STATS_DIR, fname.replace('.csv', '_stats.txt'))

        asset = detect_asset_from_filename(fname)
        if not asset:
            print(f'\n  Note: could not infer asset for {fname}, using general crypto.')
            asset = 'crypto_general'
        keywords = ASSET_KEYWORDS.get(asset, ASSET_KEYWORDS['crypto_general'])
        print(f'\n  Detected asset: {asset}')
        print(f'  Keywords: {keywords}')

        stats = clean_proquest_file(input_path, output_path, stats_path, keywords)
        if stats:
            overall_stats[fname] = stats

    # Final summary.
    print('\n' + '=' * 60)
    print('OVERALL SUMMARY')
    print('=' * 60)
    total_initial = sum(s['initial'] for s in overall_stats.values())
    total_kept = sum(s['kept'] for s in overall_stats.values())
    print(f'Files processed:      {len(overall_stats)}')
    print(f'Total input articles: {total_initial:,}')
    print(f'Total kept articles:  {total_kept:,}')
    if total_initial > 0:
        print(f'Overall retention:    {100 * total_kept / total_initial:.1f}%')

    print(f'\nPer-file results:')
    for fname, stats in overall_stats.items():
        ret = 100 * stats['kept'] / stats['initial'] if stats['initial'] else 0
        print(f'  {fname:50s} {stats["kept"]:>6,} kept ({ret:5.1f}%)')

PROCESSING 5 PROQUEST FILE(S)
  bitcoin_articles (1).csv (1.8 MB)
  crypto_general_articles.csv (0.5 MB)
  dogecoin_articles.csv (0.1 MB)
  ethereum_articles.csv (0.3 MB)
  shiba_inu_articles.csv (0.2 MB)

  Detected asset: bitcoin
  Keywords: ['bitcoin', 'btc', 'satoshi']

CLEANING: bitcoin_articles (1).csv
  Loaded: 354 articles
  Columns: ['title', 'pub_date', 'source', 'author', 'full_text', 'url', 'scrape_timestamp', 'query_name', 'month', 'date_from', 'date_to']

  --- Summary ---
  initial                             354
  dropped_extraction_failure            0
  dropped_length                       77
  dropped_date                          0
  dropped_irrelevant                    0
  dropped_duplicate                     3
  kept                                274
  retention                         77.4%
  Date range: 2020-01-01 to 2024-12-01
  Saved: data/processed/news/bitcoin_articles (1)_cleaned.csv

  Detected asset: crypto_general
  Keywords: ['crypto', 'cryptocurrenc

## 5. Download All Cleaned Files

In [ ]:
zip_name = 'proquest_cleaned'
shutil.make_archive(zip_name, 'zip', 'data/processed', 'news')
zip_path = f'{zip_name}.zip'
size_mb = os.path.getsize(zip_path) / 1e6
print(f'Archive: {zip_path} ({size_mb:.1f} MB)')
files.download(zip_path)
print('Download started.')

Archive: proquest_cleaned.zip (0.6 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.
